In [30]:
from google.colab import files
uploaded = files.upload()

Saving all_years_aggregated.csv to all_years_aggregated (1).csv


In [31]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [32]:
import pandas as pd
import numpy as np
#from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences

# revised methodology: avoids leakage from same group over time

# 1. load data
# reading in resulting CSV from aggregate_and_build.py script
# data = pd.read_csv("lstm_preprocessed_data.csv")
# data = pd.read_csv("processed/all_years_aggregated.csv")
data = pd.read_csv("all_years_aggregated.csv")

In [33]:

# 2. group id
data["group_id"] = (
    data["geo_region"].astype(str) + "_" +
    data["education_grouped"].astype(str) + "_" +
    data["age_group"].astype(str) + "_" +
    data["family_income_grouped"].astype(str)
)

# 3. survival filter
group_survival = (
    data.groupby("group_id")["year"]
    .nunique()
    .reset_index(name='year_count')
)

# making sure groups are consistent between all 8 years
# true time series will track same groups over time
surviving_groups = group_survival[group_survival["year_count"] >= 8]["group_id"]

model_data = data[data["group_id"].isin(surviving_groups)].copy()

# 4. sort
model_data = model_data.sort_values(["group_id", "year"])

# 5. fill missing
model_data = model_data.fillna(0)

In [34]:
# remove columns we already grouped on - don't need them as features! - redundant
# also werent properly encoded so were causing typecast errors
model_data = model_data.drop(columns=["age_group", "education_grouped","family_income_grouped"])

# 6. features
# everything except the group id, year, and target variable
feature_cols = [
    c for c in model_data.columns
    if c not in ["group_id", "year", "did_vote_1"]
]
# 7. buidling sequences for each group
# need to filter for years then build sequences after
train_years = 2018
val_years   = 2022
test_years  = 2024
def build_sequences(df, max_year):
    X_seq, y_seq = [], []

    for gid, g in df.groupby("group_id"):
        g = g.sort_values("year")

        g = g[g["year"] <= max_year]   # filter to find rows in corrct year

        if len(g) < 2:
            continue

        X_seq.append(g[feature_cols].values) # only rows with correct year added to sequence
        y_seq.append(g["did_vote_1"].values[-1]) # only rows with coreect year added to sequence

    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


X_train, y_train = build_sequences(model_data, 2018)
X_val, y_val     = build_sequences(model_data, 2022)
X_test, y_test   = build_sequences(model_data, 2024)



'''
# THIS IS INVALID BECAUSE HAS LEAKAGE
# split into training and testing
# this is year level split
# each year snapshot would either be in train, test, or validation set
# train on years <- 2018, validate on 2020-2022, test on 2024
train_data = model_data[model_data["year"] <= 18] # for training
val_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection
# after final model has been tuned/optimized, avoid data leakage
test_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations
'''

'\n# THIS IS INVALID BECAUSE HAS LEAKAGE\n# split into training and testing \n# this is year level split\n# each year snapshot would either be in train, test, or validation set\n# train on years <- 2018, validate on 2020-2022, test on 2024\ntrain_data = model_data[model_data["year"] <= 18] # for training\nval_data   = model_data[(model_data["year"] > 18) & (model_data["year"] <= 22)] # for tuning/selection\n# after final model has been tuned/optimized, avoid data leakage\ntest_data  = model_data[model_data["year"] == 24] # for testing performance on unseen observations \n'

In [35]:
print(X_train.shape)


(1620, 8, 59)


In [36]:
import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout

# Define input parameters
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]
# set up the architecture - subject to change
model = Sequential([
    # Input layer with input_shape matching your sequences
    LSTM(64, activation='relu', input_shape=(n_timesteps, n_features), return_sequences=False),

    # Optional: Add Dropout to prevent overfitting
    Dropout(0.2),

    # Output layer: 1 for single-value prediction, or len(y_train[0]) for multi-step
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=["mae"])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
type(X_train)

numpy.ndarray

In [38]:
type(y_train)

numpy.ndarray

In [39]:
X_train.shape

(1620, 8, 59)

In [40]:
X_train.shape[1] == 8

True

In [41]:
print(X_train.dtype)
print(type(X_train[0][0][0]))

float32
<class 'numpy.float32'>


In [42]:
print(X_train.dtype)
print(type(X_train[0]))

float32
<class 'numpy.ndarray'>


In [43]:
# Train the model
# history = model.fit(X_train, y_train, epochs=20, batch_size=256, verbose=1)

In [44]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,           # Wait 5 epochs for improvement
    restore_best_weights=True # Revert to the best model
)

# Use in model.fit
model.fit(
    X_train, y_train,
    epochs=100,           # Set a high max, let early stopping take over
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose = 1
)


Epoch 1/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 22169242497449984.0000 - mae: 82086856.0000 - val_loss: 1975916213305344.0000 - val_mae: 27142098.0000
Epoch 2/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2829006374174720.0000 - mae: 29427380.0000 - val_loss: 619978193633280.0000 - val_mae: 13402694.0000
Epoch 3/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1672760510119936.0000 - mae: 22820536.0000 - val_loss: 436338847383552.0000 - val_mae: 11677970.0000
Epoch 4/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1048746926276608.0000 - mae: 18475922.0000 - val_loss: 301120526221312.0000 - val_mae: 8706236.0000
Epoch 5/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 889322504978432.0000 - mae: 15684593.0000 - val_loss: 224218163707904.0000 - val_mae: 7570345.0000
Epoch 6/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 581746273812480.0000 - mae: 13259016.0000 - val_loss: 155274040049664.0000 - val_mae: 6244885.5000
Epoch 7/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [45]:
val_loss, val_mae = model.evaluate(X_val, y_val)
print("Validation Loss:", val_loss)
print("Validation MAE:", val_mae)

51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 248680775680.0000 - mae: 244586.0000
Validation Loss: 248680775680.0
Validation MAE: 244586.0


In [46]:
model.metrics_names

['loss', 'compile_metrics']

In [47]:
y_pred = model.predict(X_val)

51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step


In [54]:
y_pred.shape

(1620,)

In [55]:
y_val.shape

(1620,)

In [56]:
y_pred = y_pred.flatten()
y_true = y_val.flatten()

In [57]:
y_true.shape

(1620,)

In [58]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

MAE: 244586.0
MSE: 248680808448.0
RMSE: 498679.063574961


In [59]:
from sklearn.metrics import r2_score

r2 = r2_score(y_true, y_pred)
print("R²:", r2)

R²: -3760986783744.0
